In [22]:
from langchain_core.tools import tool 
from langchain.agents import create_agent 
from langchain_aws import ChatBedrockConverse

In [23]:
llm = ChatBedrockConverse(model ="cohere.command-r-plus-v1:0", region_name = 'us-east-1', temperature = 0.4, max_tokens = 300)

In [24]:
@tool 
def add_numbers(a: int, b:int) -> int: 
    """ Add two numbers and return the result """ 
    return a + b 

@tool 
def square_number(x: int) -> int: 
    """ Return the square of the number """ 
    return x * x

In [25]:
llm_with_tools = llm.bind_tools([add_numbers, square_number])

In [26]:
query = "what is squared result of 5 plus 10. Show the result ?" 
llm_with_tools.invoke(query)

AIMessage(content=[{'type': 'text', 'text': 'Here is how to calculate the squared result of 5 plus 10:'}, {'type': 'tool_use', 'name': 'add_numbers', 'input': {'a': 5, 'b': 10}, 'id': 'tooluse_XvjAQbGN3N0w5ubCVYTrPq'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '13d95ad0-9a07-4186-9611-800fb3bb921a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 17:10:18 GMT', 'content-type': 'application/json', 'content-length': '365', 'connection': 'keep-alive', 'x-amzn-requestid': '13d95ad0-9a07-4186-9611-800fb3bb921a'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [1315]}, 'model_provider': 'bedrock_converse', 'model_name': 'anthropic.claude-3-sonnet-20240229-v1:0'}, id='lc_run--019d9745-a458-7f10-8689-b1691e1634eb-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 5, 'b': 10}, 'id': 'tooluse_XvjAQbGN3N0w5ubCVYTrPq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 304, 'output_tokens': 87

In [18]:
chain = llm_with_tools | (lambda x: x.tool_calls[0]['args']) | add_numbers | (lambda result: {"x":result}) | square_number

In [19]:
chain.invoke("What is the square of the sum of 4 and 5 ?")

81

In [20]:
tools = [add_numbers, square_number] 
agent = create_agent(model = llm, tools = tools, system_prompt = "You are a mathematical agent")

In [21]:
agent.invoke({'messages':[("user","Square 55 and add 33 with 22 and find the sqaure of the result")]})

{'messages': [HumanMessage(content='Square 55 and add 33 with 22 and find the sqaure of the result', additional_kwargs={}, response_metadata={}, id='5d20e8e3-12d9-4498-98a0-98fd8daa33e6'),
  AIMessage(content=[{'type': 'text', 'text': 'I will first square 55, then add 33 and 22, and then square the result.'}, {'type': 'tool_use', 'name': 'square_number', 'input': {'x': 55}, 'id': 'tooluse_iZryqhrwAmxyKnTUmhlTX3'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'ebc33f43-7777-4201-acf0-bc12a80f6cbb', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 17:07:25 GMT', 'content-type': 'application/json', 'content-length': '373', 'connection': 'keep-alive', 'x-amzn-requestid': 'ebc33f43-7777-4201-acf0-bc12a80f6cbb'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [2034]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'}, id='lc_run--019d9742-fd6e-7f10-9ca9-84bdea7cc896-0', tool_calls=[{'name'

In [28]:
#Example-2 
@tool 
def analyze_cardio(player_id:str, curr_hr:int): 
    """ Calculate if a player's Hr is exceeding 90% of the threshold (200 BPM max) """ 
    max_hr = 200 
    threshold = 0.9 * 200 
    strain_level = "CRITICAL" if curr_hr > threshold else "STABLE" 
    diff = curr_hr - threshold 
    return{
        'player_id':player_id, 
        'status':strain_level, 
        'bpm_ovr_thresh':max(0,diff), 
        'recommendation': "Immediate rest is recommended" if strain_level == 'CRITICAL' else "Continue to play"
    }

In [37]:
@tool 
def calculate_speed_eff(player_id: str, current_speed : float, target_speed:float): 
    """ Calculate the speed efficieny against the target speed set """ 
    efficiency = (current_speed / target_speed) * 100 
    return { 
    'player_id':player_id,
    'efficieny_score':f"{round(efficiency,1)}%",
    'is_efficient':efficiency > 85.0, 
    "gap": f"{round(target_speed - current_speed,1)} km/h"
    }

tools = [calculate_speed_eff, analyze_cardio]

In [38]:
agent = create_agent(model = llm, tools = tools, system_prompt = "You are a personal health threshold monitor agent for soccer players")

In [39]:
agent.invoke({'messages':'analyze player 18#, their heart rate is 188 bpm and theor current speed is 15 km/h. I want them to achieve 20 km/h. Provide a status report on this'})

{'messages': [HumanMessage(content='analyze player 18#, their heart rate is 188 bpm and theor current speed is 15 km/h. I want them to achieve 20 km/h. Provide a status report on this', additional_kwargs={}, response_metadata={}, id='3b4b1c57-d5e7-412f-be8a-f2ee5f1e25ec'),
  AIMessage(content=[{'type': 'text', 'text': "Okay, let's analyze the situation for player 18#:"}, {'type': 'tool_use', 'name': 'analyze_cardio', 'input': {'player_id': '18#', 'curr_hr': 188}, 'id': 'tooluse_TJvkzqnMtgBZFOsP9rSnVs'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'f0e9ea12-6e5d-40f5-bd52-1a8646027100', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 17:44:01 GMT', 'content-type': 'application/json', 'content-length': '379', 'connection': 'keep-alive', 'x-amzn-requestid': 'f0e9ea12-6e5d-40f5-bd52-1a8646027100'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [1373]}, 'model_provider': 'bedrock_converse', 'model_name': 'anthropic.c